In [1]:
import os
import sys
import gc
import json
import math
import time
import random
import re
import types
import subprocess
import importlib.util
import importlib.machinery
import contextlib
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# ============================================================
# Bootstrap / install
# ============================================================

def _stub_torchaudio() -> None:
    """Avoid a broken torchaudio binary from breaking Transformers imports."""
    if "torchaudio" in sys.modules:
        return
    ta = types.ModuleType("torchaudio")
    ta.__dict__["__version__"] = "0.0.0"
    ta.__spec__ = importlib.machinery.ModuleSpec("torchaudio", loader=None)
    ta.__path__ = []
    sys.modules["torchaudio"] = ta
    for sm in ["_extension", "datasets", "functional", "models", "pipelines", "transforms", "utils"]:
        m = types.ModuleType(f"torchaudio.{sm}")
        m.__spec__ = importlib.machinery.ModuleSpec(f"torchaudio.{sm}", loader=None)
        m.__path__ = []
        setattr(ta, sm, m)
        sys.modules[f"torchaudio.{sm}"] = m


_stub_torchaudio()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "true")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")


def ensure_pkg(module_name: str, pip_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing missing dependency: {pip_name}", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", pip_name])


# Keep dependencies modest and Kaggle-safe.
for mod, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("datasets", "datasets"),
    ("transformers", "transformers<=4.48.3"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
    ("sentencepiece", "sentencepiece"),
    ("bitsandbytes", "bitsandbytes"),
]:
    try:
        ensure_pkg(mod, pip_name)
    except Exception as e:
        print(f"Dependency bootstrap warning for {pip_name}: {e}", flush=True)

from datasets import load_dataset
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training

# Optional plotting extras.
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "font.family": "DejaVu Sans",
})


# ============================================================
# Cache Bug Fix for Native Phi-3 Inference
# ============================================================
try:
    from transformers.cache_utils import DynamicCache
    if not hasattr(DynamicCache, "from_legacy_cache"):
        @classmethod
        def _from_legacy_cache(cls, past_key_values=None):
            return cls()
        DynamicCache.from_legacy_cache = _from_legacy_cache
        
    if not hasattr(DynamicCache, "get_usable_length"):
        def _get_usable_length(self, new_seq_length=None, layer_hash=None):
            return getattr(self, "get_seq_length", lambda: 0)()
        DynamicCache.get_usable_length = _get_usable_length
except Exception:
    pass

if hasattr(torch, "compiler") and hasattr(torch.compiler, "disable"):
    _original_disable = torch.compiler.disable
    def _safe_disable(fn=None, recursive=True, **kwargs):
        kwargs.pop("reason", None) 
        return _original_disable(fn=fn, recursive=recursive)
    torch.compiler.disable = _safe_disable

# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    base_model: str = "microsoft/phi-3-mini-4k-instruct"
    start_adapter: Optional[str] = "/kaggle/input/datasets/adithyaled24b039/phi3-sft-folderr/phi3_sft_lora"
    hard_csv: str = "/mnt/data/all_predictions_all_models.csv"
    out_dir: str = "/kaggle/working/phi3_rrpo_hardmine"

    seed: int = 42
    use_4bit: bool = True
    bf16: bool = False
    fp16: bool = True
    gradient_checkpointing: bool = True

    # Training budget (adjust upward if you have time).
    sft_steps: int = 220
    pairwise_steps: int = 180
    rrpo_steps: int = 260
    eval_every: int = 40
    save_every: int = 60

    # Batch / sampling.
    micro_batch_size: int = 1
    grad_accum: int = 4
    num_candidates: int = 4
    max_new_gsm8k: int = 160
    max_new_qa: int = 24
    max_new_mmlu: int = 20
    self_consistency_gsm8k: int = 3
    self_consistency_qa: int = 3
    self_consistency_mmlu: int = 1

    # Data sizes.
    gsm8k_train: int = 1200
    strategyqa_train: int = 900
    mmlu_train_per_subject: int = 28
    gsm8k_eval: int = 50
    strategyqa_eval: int = 50
    mmlu_eval_per_subject: int = 12
    fewshot_gsm8k: int = 2
    fewshot_qa: int = 2
    fewshot_mmlu: int = 3

    # LoRA.
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05

    # Optimization.
    lr: float = 5e-5
    wd: float = 0.01
    max_grad_norm: float = 1.0
    warmup_ratio: float = 0.05

    # Loss weights.
    w_supervised: float = 1.0
    w_routing: float = 0.35
    w_margin: float = 0.15
    w_pairwise: float = 1.0
    w_reward_margin: float = 0.08
    w_rl_entropy: float = 0.02
    w_rl_hard: float = 0.08
    w_kl_approx: float = 0.0  # kept off by default; policy ratio/clipping is used instead.

    # PPO/GRPO-like clipping.
    clip_eps: float = 0.2
    entropy_beta: float = 0.01

    # Generation.
    temperature_start: float = 0.95
    temperature_end: float = 0.65
    top_p: float = 0.92
    repetition_penalty: float = 1.02

    # Prompting.
    include_fewshot: bool = True
    use_chat_template: bool = True
    max_prompt_tokens: int = 1536

    # Subject pool.
    mmlu_subjects: Tuple[str, ...] = (
        "abstract_algebra",
        "college_mathematics",
        "logical_fallacies",
        "formal_logic",
        "high_school_mathematics",
    )

    # Evaluation.
    eval_majority_vote: bool = True
    eval_use_random_sampling: bool = True


CFG = Config()

ROUTES = ["direct", "reason", "selfcheck"]

# ============================================================
# Logging / dirs
# ============================================================

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

DEVICE_COUNT = torch.cuda.device_count()
POLICY_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
REF_DEVICE = torch.device("cuda:1" if torch.cuda.device_count() > 1 else "cpu")
DTYPE = torch.float16 if CFG.fp16 and torch.cuda.is_available() else torch.float32

OUT_DIR = CFG.out_dir
CSV_DIR = os.path.join(OUT_DIR, "csv")
PLOT_DIR = os.path.join(OUT_DIR, "plots")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
REPORT_DIR = os.path.join(OUT_DIR, "reports")
CACHE_DIR = os.path.join(OUT_DIR, "cache")
for p in [OUT_DIR, CSV_DIR, PLOT_DIR, CKPT_DIR, REPORT_DIR, CACHE_DIR]:
    os.makedirs(p, exist_ok=True)


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def gpu_report(prefix: str = "GPU") -> None:
    if not torch.cuda.is_available():
        log("CUDA unavailable.")
        return
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / (1024**3)
        reserv = torch.cuda.memory_reserved(i) / (1024**3)
        log(f"{prefix} {i}: allocated={alloc:.2f} GB reserved={reserv:.2f} GB")


# ============================================================
# Utilities
# ============================================================

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.synchronize()
        except Exception:
            pass


def save_df(df: pd.DataFrame, path: str):
    df.to_csv(path, index=False)
    log(f"Saved CSV -> {path}")


def save_json(obj: Any, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    log(f"Saved JSON -> {path}")


def save_plot(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    log(f"Saved plot -> {path}")


def timer(label: str):
    class _T:
        def __enter__(self_inner):
            self_inner.t0 = time.time()
            log(f"{label} ...")
            return self_inner
        def __exit__(self_inner, exc_type, exc, tb):
            dt = time.time() - self_inner.t0
            if exc_type is None:
                log(f"{label} done in {dt:.1f}s")
            else:
                log(f"{label} failed after {dt:.1f}s: {exc_type.__name__}: {exc}")
    return _T()


@contextlib.contextmanager
def evaluate_mode(model):
    """
    Ensures LoRA dropout is completely disabled during generation rollouts and evaluations.
    Restores the model to its prior training state afterward.
    """
    is_training = model.training
    model.eval()
    try:
        yield
    finally:
        if is_training:
            model.train()


def sample_indices(n: int, k: int, seed: int, cache_name: str) -> List[int]:
    k = min(k, n)
    path = os.path.join(CACHE_DIR, f"{cache_name}_{k}_{seed}.json")
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    rng = np.random.RandomState(seed)
    idx = rng.choice(n, size=k, replace=False).tolist()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(idx, f)
    return idx


def sample_dataset(ds, k: int, seed: int, cache_name: str):
    idx = sample_indices(len(ds), k, seed, cache_name)
    return ds.select(idx), idx


def preview_text(s: str, n: int = 120) -> str:
    s = str(s).replace("\n", " ")
    return s[:n] + ("..." if len(s) > n else "")

# ============================================================
# Plotting Functions
# ============================================================

def annotated_bar(ax, fmt="{:.3f}"):
    for p in ax.patches:
        h = p.get_height()
        if np.isfinite(h) and h > 0:
            ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2.0, h),
                        ha="center", va="bottom", fontsize=8, xytext=(0, 3), textcoords="offset points")

def bar_plot(labels, values, title, path, ylabel="Accuracy", rotate=0, color_map="Set2"):
    if not labels: return
    plt.figure(figsize=(10.5, 5.5))
    ax = plt.gca()
    colors = plt.get_cmap(color_map)(np.linspace(0.12, 0.88, len(labels)))
    ax.bar(labels, values, color=colors)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotate)
    ax.set_ylim(0, max(1.0, max(values) + 0.1))
    annotated_bar(ax)
    save_plot(path)

def plot_route_distribution(df: pd.DataFrame, path: str, title: str = "Route distribution by task") -> None:
    if len(df) == 0 or "route_pred" not in df.columns:
        return
    tab = pd.crosstab(df["task"], df["route_pred"].fillna("missing"), normalize="index")
    tab = tab.reindex(columns=ROUTES).fillna(0.0)
    heatmap = tab.values
    
    plt.figure(figsize=(8.8, 4.8))
    im = plt.imshow(heatmap, aspect="auto", vmin=0, vmax=1.0, cmap="Blues")
    plt.colorbar(im)
    plt.xticks(range(len(tab.columns)), tab.columns)
    plt.yticks(range(len(tab.index)), tab.index)
    plt.title(title)
    
    for i in range(heatmap.shape[0]):
        for j in range(heatmap.shape[1]):
            plt.text(j, i, f"{heatmap[i, j]:.2f}", ha="center", va="center", 
                     color="white" if heatmap[i, j] > 0.5 else "black")
    save_plot(path)

def plot_length_hist(df: pd.DataFrame, path: str, title: str) -> None:
    if len(df) == 0 or "completion_length" not in df.columns:
        return
    plt.figure(figsize=(8.5, 4.8))
    for task, g in df.groupby("task"):
        plt.hist(g["completion_length"], bins=25, alpha=0.5, label=task)
    plt.title(title)
    plt.xlabel("Tokens / Characters")
    plt.ylabel("Count")
    plt.legend(frameon=True)
    save_plot(path)

def violin_plot_lengths(df: pd.DataFrame, path: str):
    if len(df) == 0 or "completion_length" not in df.columns:
        return
    plt.figure(figsize=(10.5, 6))
    tasks = sorted(df["task"].unique())
    data = [df[df["task"] == t]["completion_length"].dropna().to_numpy(dtype=np.float64) for t in tasks]
    valid_idx = [i for i, d in enumerate(data) if len(d) > 0]
    
    if valid_idx:
        parts = plt.violinplot([data[i] for i in valid_idx], showmeans=True)
        for pc in parts['bodies']:
            pc.set_facecolor('tab:blue')
            pc.set_edgecolor('black')
            pc.set_alpha(0.6)
        plt.xticks(range(1, len(valid_idx) + 1), [tasks[i] for i in valid_idx], rotation=0)
        
    plt.title("Distribution of Completion Lengths (Characters) by Task")
    plt.ylabel("Completion Length")
    save_plot(path)

def plot_training_curves(hist: pd.DataFrame, path: str):
    if len(hist) == 0:
        return
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    if "loss" in hist:
        axes[0, 0].plot(hist["step"], hist["loss"], label="loss")
        axes[0, 0].set_title("Loss")
    if "reward" in hist:
        axes[0, 1].plot(hist["step"], hist["reward"], label="reward")
        axes[0, 1].set_title("Reward")
    if "route" in hist:
        axes[1, 0].plot(hist["step"], hist["route"], label="route")
        axes[1, 0].set_title("Routing loss")
    if "eval_acc" in hist:
        axes[1, 1].plot(hist["step"], hist["eval_acc"], label="eval_acc")
        axes[1, 1].set_title("Eval accuracy")
    for ax in axes.flat:
        ax.legend(loc="best")
    save_plot(path)

def plot_accuracy_progress(progress_rows: pd.DataFrame, path: str):
    if len(progress_rows) == 0:
        return
    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    for task in progress_rows["task"].unique():
        sub = progress_rows[progress_rows.task == task].sort_values("stage_idx")
        ax.plot(sub["stage_name"], sub["accuracy"], marker="o", linewidth=2, label=task)
        for _, r in sub.iterrows():
            ax.text(r["stage_name"], r["accuracy"] + 0.005, f"{r['accuracy']:.2f}", fontsize=8, ha="center")
    ax.set_ylim(0, 1.05)
    ax.set_title("Task accuracy progression over checkpoints")
    ax.set_xlabel("Stage")
    ax.set_ylabel("Accuracy")
    ax.legend()
    save_plot(path)


# ============================================================
# Parsing / evaluation helpers
# ============================================================

def canonical_number(pred: Optional[str]) -> Optional[str]:
    if pred is None:
        return None
    try:
        x = float(str(pred).replace(",", ""))
        return str(int(round(x))) if abs(x - round(x)) < 1e-8 else str(x)
    except Exception:
        return None

def is_number_correct(pred: Optional[str], gold: Optional[str]) -> bool:
    try:
        return abs(float(pred) - float(gold)) <= 1e-6
    except Exception:
        return False

def extract_route(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<route>\s*(direct|reason|selfcheck)\s*</route>", text, flags=re.IGNORECASE)
    if m:
        return m[-1].lower()
    m = re.findall(r"\b(direct|reason|selfcheck)\b", text, flags=re.IGNORECASE)
    return m[-1].lower() if m else None

def extract_number(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    boxed = re.findall(r"\\boxed\{([^}]*)\}", span)
    if boxed:
        nums = re.findall(r"-?\d+\.?\d*", boxed[-1].replace(",", ""))
        if nums:
            return nums[-1]
    nums = re.findall(r"-?\d+\.?\d*", span.replace(",", ""))
    return nums[-1] if nums else None

def extract_yes_no(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    hits = re.findall(r"\b(yes|no)\b", span, flags=re.IGNORECASE)
    if hits:
        return hits[-1].lower()
    hits = re.findall(r"\b(yes|no)\b", text, flags=re.IGNORECASE)
    return hits[-1].lower() if hits else None


def extract_mcq(text: str) -> Optional[str]:
    if not text:
        return None
    m = re.findall(r"<answer>(.*?)</answer>", text, flags=re.DOTALL | re.IGNORECASE)
    span = m[-1] if m else text
    up = span.upper().strip()
    if up in ["A", "B", "C", "D"]:
        return up
    for pat in [r"ANSWER\s*[:\-]?\s*\(?([ABCD])\)?", r"<ANSWER>\s*\(?([ABCD])\)?", r"\b([ABCD])\b\s*$"]:
        hits = re.findall(pat, up)
        if hits:
            return hits[-1].upper()
    hits = re.findall(r"\b([ABCD])\b", up)
    return hits[-1].upper() if hits else None


def answer_present(text: str) -> bool:
    return bool(re.search(r"\b(yes|no|[ABCD])\b", str(text), flags=re.IGNORECASE))


def normalize_choice_order(choices: List[str], perm: Tuple[int, int, int, int]):
    return [choices[i] for i in perm]


def canonical_letter_from_perm_answer(pred_letter: Optional[str], perm: Tuple[int, int, int, int]) -> Optional[str]:
    if pred_letter is None:
        return None
    pred_letter = pred_letter.upper().strip()
    if pred_letter not in ["A", "B", "C", "D"]:
        return None
    idx = ord(pred_letter) - 65
    return chr(65 + perm[idx])


def get_mmlu_choices(sample: Dict[str, Any]) -> List[str]:
    choices = sample.get("choices")
    if choices is None:
        return []
    if isinstance(choices, str):
        try:
            parsed = json.loads(choices)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return list(choices)


def safe_mmlu_gold(sample: Dict[str, Any]) -> Optional[str]:
    g = sample.get("gold")
    if g is None:
        return None
    g = str(g).strip().upper()
    return g if g in ["A", "B", "C", "D"] else None


def choice_ids(tokenizer, txt: str) -> List[int]:
    ids = []
    for form in [txt, f" {txt}", f"\n{txt}", txt.upper(), f" {txt.upper()}"]:
        enc = tokenizer.encode(form, add_special_tokens=False)
        if len(enc) == 1:
            ids.append(enc[0])
    if not ids:
        enc = tokenizer.encode(txt, add_special_tokens=False)
        if enc:
            ids.append(enc[0])
    return sorted(set(ids))


def topn_token_strings(tokenizer, logits: torch.Tensor, n: int = 5):
    probs = torch.softmax(logits.float(), dim=-1)
    vals, ids = torch.topk(probs, k=min(n, probs.shape[-1]))
    return [(tokenizer.decode([i]).strip(), float(p)) for p, i in zip(vals.tolist(), ids.tolist())]


def build_chat_prompt(tokenizer, system: str, user: str) -> str:
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return f"<|system|>\n{system}\n<|user|>\n{user}\n<|assistant|>\n"


def build_task_prompt(tokenizer, sample: Dict[str, Any], fewshots: List[Dict[str, Any]]) -> str:
    task = sample["task"]
    system = "You are a careful reasoning assistant. Think privately and output the final answer in the required format."
    fs = ""
    
    if fewshots:
        blocks = []
        for ex in fewshots:
            if ex["task"] == "gsm8k":
                blocks.append(f"Question: {ex['question']}\n<route>{ex.get('preferred_mode', 'reason')}</route><think>Thinking process...</think><answer>{ex['gold']}</answer>")
            elif ex["task"] == "strategyqa":
                blocks.append(f"Question: {ex['question']}\n<route>{ex.get('preferred_mode', 'direct')}</route><think>Thinking process...</think><answer>{ex['gold']}</answer>")
            elif ex["task"] == "mmlu":
                choice_txt = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(ex.get('choices', ['A', 'B', 'C', 'D'])[:4])])
                blocks.append(f"Question: {ex['question']}\n{choice_txt}\n<route>{ex.get('preferred_mode', 'direct')}</route><think>Thinking process...</think><answer>{ex['gold']}</answer>")
        fs = "\n\n".join(blocks) + "\n\n"

    if task == "gsm8k":
        user = (
            "You are a routing reasoning agent.\n"
            "First choose a route from {direct, reason, selfcheck}.\n"
            "Then solve the problem with the route you chose.\n"
            "Output exactly: <route>...</route><think>...</think><answer>...</answer>\n"
            "For GSM8K, put the final numeric answer inside <answer> and prefer \\boxed{answer}.\n\n"
            f"{fs}Question: {sample['question']}\n"
        )
    elif task == "strategyqa":
        user = (
            "You are a routing reasoning agent.\n"
            "First choose a route from {direct, reason, selfcheck}.\n"
            "Then answer the yes/no question.\n"
            "Output exactly: <route>...</route><think>...</think><answer>yes/no</answer>\n\n"
            f"{fs}Question: {sample['question']}\n"
        )
    elif task == "mmlu":
        choice_txt = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(sample.get('choices', ['A', 'B', 'C', 'D'])[:4])])
        user = (
            "You are a routing reasoning agent.\n"
            "First choose a route from {direct, reason, selfcheck}.\n"
            "Then answer the multiple choice question.\n"
            "Output exactly: <route>...</route><think>...</think><answer>A/B/C/D</answer>\n\n"
            f"{fs}Question: {sample['question']}\n{choice_txt}\n"
        )
    else:
        raise ValueError(task)
        
    return build_chat_prompt(tokenizer, system, user)


def parse_prediction(sample: Dict[str, Any], completion: str, mmlu_perm: Optional[Tuple[int, int, int, int]] = None):
    task = sample["task"]
    if task == "gsm8k":
        pred = canonical_number(extract_number(completion))
        return pred, bool(is_number_correct(pred, sample["gold"]))
    if task == "strategyqa":
        pred = extract_yes_no(completion)
        return pred, bool(pred == sample["gold"])
    if task == "mmlu":
        pred = extract_mcq(completion)
        if pred is None:
            return None, False
        if mmlu_perm is not None:
            pred = canonical_letter_from_perm_answer(pred, mmlu_perm)
        return pred, bool(pred == sample["gold"])
    raise ValueError(task)


# ============================================================
# Random sampling / hard mining
# ============================================================

MMLU_SUBJECTS = list(CFG.mmlu_subjects)


def load_error_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
        if not {"task", "sample_id", "question", "gold", "model", "correct"}.issubset(df.columns):
            return pd.DataFrame()
        return df
    except Exception:
        return pd.DataFrame()


def mine_hard_examples(error_df: pd.DataFrame) -> pd.DataFrame:
    if error_df is None or len(error_df) == 0:
        return pd.DataFrame(columns=["task", "sample_id", "question", "gold", "hard_score", "persistent_wrong"])
    grp = error_df.groupby(["task", "sample_id", "question", "gold"], as_index=False).agg(
        mean_correct=("correct", "mean"),
        n_models=("correct", "count"),
    )
    grp["wrong_rate"] = 1.0 - grp["mean_correct"]
    grp["persistent_wrong"] = grp["mean_correct"] <= 0.0
    grp["hard_score"] = grp["wrong_rate"] + grp["persistent_wrong"].astype(float) * 0.75
    return grp.sort_values(["hard_score", "wrong_rate"], ascending=False).reset_index(drop=True)


# ============================================================
# Dataset loading
# ============================================================

def load_gsm8k_split(split: str, n: int, cache_name: str):
    for repo in ["openai/gsm8k", "madrylab/gsm8k-platinum"]:
        try:
            ds = load_dataset(repo, "main", split=split)
            ds, idx = sample_dataset(ds, n, CFG.seed, cache_name + f"_{repo.replace('/', '_')}")
            rows = []
            for i, s in enumerate(ds):
                rows.append({
                    "task": "gsm8k",
                    "subject": "gsm8k",
                    "sample_id": f"gsm8k_{split}_{idx[i]}",
                    "source_index": idx[i],
                    "question": s["question"],
                    "gold": canonical_number(str(s["answer"]).split("####")[-1].strip()),
                    "raw_gold": s["answer"],
                    "model": None,
                    "full_answer": s["answer"],
                    "source_repo": repo,
                })
            return pd.DataFrame(rows)
        except Exception:
            continue
    return pd.DataFrame()


def load_strategyqa_split(split: str, n: int, cache_name: str):
    try:
        ds = load_dataset("ChilleD/StrategyQA", split=split)
        ds, idx = sample_dataset(ds, n, CFG.seed, cache_name)
        rows = []
        for i, s in enumerate(ds):
            rows.append({
                "task": "strategyqa",
                "sample_id": f"strategyqa_{split}_{idx[i]}",
                "source_index": idx[i],
                "question": s["question"],
                "gold": "yes" if bool(s["answer"]) else "no",
                "raw_gold": s["answer"],
                "model": None,
            })
        return pd.DataFrame(rows)
    except Exception:
        return pd.DataFrame()


def load_mmlu_subject_split(subject: str, split_candidates: Tuple[str, ...], n: int, cache_name: str):
    for split in split_candidates:
        try:
            ds = load_dataset("cais/mmlu", subject, split=split)
            ds, idx = sample_dataset(ds, n, CFG.seed, f"{cache_name}_{subject}_{split}")
            rows = []
            for i, s in enumerate(ds):
                choices = get_mmlu_choices(s)
                gold = safe_mmlu_gold({"gold": chr(65 + int(s["answer"])) if isinstance(s["answer"], (int, np.integer)) else str(s["answer"])})
                if gold is None and isinstance(s["answer"], (int, np.integer)):
                    gold = chr(65 + int(s["answer"]))
                rows.append({
                    "task": "mmlu",
                    "subject": subject,
                    "sample_id": f"mmlu_{subject}_{split}_{idx[i]}",
                    "source_index": idx[i],
                    "question": s["question"],
                    "choices": choices[:4],
                    "gold": gold,
                    "raw_gold": s["answer"],
                    "mmlu_split": split,
                    "model": None,
                })
            return pd.DataFrame(rows)
        except Exception:
            continue
    return pd.DataFrame()


def build_task_pools():
    log("Building task pools ...")
    pools = {}

    # Train pools (larger, randomized, oversampled from hard failures).
    pools["gsm_train"] = load_gsm8k_split("train", CFG.gsm8k_train, "gsm8k_train")
    pools["qa_train"] = load_strategyqa_split("train", CFG.strategyqa_train, "strategyqa_train")

    mmlu_parts = []
    for subject in MMLU_SUBJECTS:
        part = load_mmlu_subject_split(subject, ("dev", "validation", "train", "test"), CFG.mmlu_train_per_subject, f"mmlu_train")
        if len(part) > 0:
            mmlu_parts.append(part)
    pools["mmlu_train"] = pd.concat(mmlu_parts, ignore_index=True) if mmlu_parts else pd.DataFrame()

    # Eval pools (random held out subsets, not first-N).
    pools["gsm_eval"] = load_gsm8k_split("test", CFG.gsm8k_eval, "gsm8k_eval")
    pools["qa_eval"] = load_strategyqa_split("test", CFG.strategyqa_eval, "strategyqa_eval")

    mmlu_eval_parts = []
    for subject in MMLU_SUBJECTS:
        part = load_mmlu_subject_split(subject, ("test", "validation", "dev"), CFG.mmlu_eval_per_subject, f"mmlu_eval")
        if len(part) > 0:
            mmlu_eval_parts.append(part)
    pools["mmlu_eval"] = pd.concat(mmlu_eval_parts, ignore_index=True) if mmlu_eval_parts else pd.DataFrame()

    return pools


# ============================================================
# Prompt rendering / few-shots
# ============================================================

def build_fewshot_bank(train_df: pd.DataFrame):
    bank = {"gsm8k": [], "strategyqa": [], "mmlu": {}}
    if len(train_df) == 0:
        return bank

    gsm = train_df[train_df.task == "gsm8k"].sample(min(40, (train_df.task == "gsm8k").sum()), random_state=CFG.seed)
    qa = train_df[train_df.task == "strategyqa"].sample(min(40, (train_df.task == "strategyqa").sum()), random_state=CFG.seed)
    mmlu = train_df[train_df.task == "mmlu"].sample(min(40, (train_df.task == "mmlu").sum()), random_state=CFG.seed)
    bank["gsm8k"] = gsm.to_dict("records")
    bank["strategyqa"] = qa.to_dict("records")
    for subj, g in mmlu.groupby("subject"):
        bank["mmlu"][subj] = g.to_dict("records")
    return bank

def choose_fewshots(sample: Dict[str, Any], bank: Dict[str, Any], k: int) -> List[Dict[str, Any]]:
    if k <= 0:
        return []
    task = sample["task"]
    
    # Get appropriate pool
    if task == "gsm8k":
        pool = bank.get("gsm8k", [])
    elif task == "strategyqa":
        pool = bank.get("strategyqa", [])
    else:
        subj = sample.get("subject", "")
        pool = bank.get("mmlu", {}).get(subj, [])
        if not pool and bank.get("mmlu"):
            pool = sum(bank["mmlu"].values(), [])
            
    if not pool:
        return []
        
    # Filter out the sample itself
    pool = [x for x in pool if x.get("sample_id") != sample.get("sample_id")]
    if not pool:
        return []
        
    # Fast, non-IO random sampling to avoid generating 100,000s of cache files
    seed = CFG.seed + abs(hash(str(sample.get("sample_id", "")))) % 10000
    rng = random.Random(seed)
    return rng.sample(pool, min(k, len(pool)))

def add_hard_examples(train_df: pd.DataFrame, hard_df: pd.DataFrame) -> pd.DataFrame:
    if len(hard_df) == 0:
        return train_df
    # Pull persistent failures and high-hardness rows into the training mix.
    hard = hard_df.copy().sort_values(["hard_score"], ascending=False)
    if "persistent_wrong" in hard.columns:
        hard = hard[hard["persistent_wrong"] | (hard["hard_score"] > 0.4)]
    if len(hard) == 0:
        return train_df

    rows = []
    for _, r in hard.iterrows():
        task = r["task"]
        if task == "gsm8k":
            rows.append({
                "task": "gsm8k",
                "sample_id": f"hard_{r['sample_id']}",
                "question": r["question"],
                "gold": str(r["gold"]),
                "raw_gold": r["gold"],
                "source": "hard_csv",
                "hard_score": float(r["hard_score"]),
                "subject": "gsm8k",
            })
        elif task == "strategyqa":
            rows.append({
                "task": "strategyqa",
                "sample_id": f"hard_{r['sample_id']}",
                "question": r["question"],
                "gold": str(r["gold"]),
                "raw_gold": r["gold"],
                "source": "hard_csv",
                "hard_score": float(r["hard_score"]),
                "subject": "strategyqa",
            })
        # Skip MMLU CSV hard rows because the uploaded CSV does not preserve choices.
    hard_df2 = pd.DataFrame(rows)
    if len(hard_df2) == 0:
        return train_df
    repeats = []
    for _, r in hard_df2.iterrows():
        rep = 2 + int(r.get("hard_score", 0) > 0.8)
        repeats.extend([r.to_dict()] * rep)
    hard_os = pd.DataFrame(repeats)
    merged = pd.concat([train_df, hard_os], ignore_index=True)
    return merged.sample(frac=1.0, random_state=CFG.seed).reset_index(drop=True)


def render_target_completion(sample: Dict[str, Any]) -> str:
    if sample["task"] == "gsm8k":
        raw = str(sample.get("raw_gold", ""))
        return raw.strip() if raw.strip() else f"<answer>{sample['gold']}</answer>"
    if sample["task"] == "strategyqa":
        return f"<answer>{sample['gold']}</answer>"
    if sample["task"] == "mmlu":
        return f"<answer>{sample['gold']}</answer>"
    raise ValueError(sample["task"])


def render_prompt(tokenizer, sample: Dict[str, Any], fewshots: List[Dict[str, Any]]) -> str:
    return build_task_prompt(tokenizer, sample, fewshots)


# ============================================================
# Tokenization / model helpers
# ============================================================

def discover_lora_targets(model) -> List[str]:
    # Phi-3 naming seen in HF remote code: qkv_proj / o_proj / gate_up_proj / down_proj.
    candidates = ["qkv_proj", "o_proj", "gate_up_proj", "up_proj", "down_proj"]
    names = set()
    for n, m in model.named_modules():
        if isinstance(m, torch.nn.Linear):
            leaf = n.split(".")[-1]
            if leaf in candidates:
                names.add(leaf)
    return sorted(names) if names else ["qkv_proj", "o_proj", "gate_up_proj", "up_proj", "down_proj"]


def normalize_rope_scaling(cfg):
    rs = getattr(cfg, "rope_scaling", None)
    if isinstance(rs, dict):
        if "type" not in rs and "rope_type" in rs:
            rs["type"] = rs["rope_type"]
        if "rope_type" not in rs and "type" in rs:
            rs["rope_type"] = rs["type"]
        cfg.rope_scaling = rs
    return cfg


global tokenizer

def load_tokenizer():
    log(f"Loading tokenizer: {CFG.base_model}")
    tok = AutoTokenizer.from_pretrained(CFG.base_model, trust_remote_code=False)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def load_base_model(device: torch.device):
    log(f"Loading base model on {device} ...")
    load_kwargs = dict(
        low_cpu_mem_usage=True,
        torch_dtype=DTYPE,
        attn_implementation="eager",
        device_map={"": device.index if device.type == "cuda" else "cpu"},
    )
    if CFG.use_4bit and torch.cuda.is_available() and importlib.util.find_spec("bitsandbytes") is not None:
        qcfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16 if CFG.fp16 else torch.bfloat16,
        )
        load_kwargs["quantization_config"] = qcfg
    try:
        cfg = AutoConfig.from_pretrained(CFG.base_model, trust_remote_code=False)
        cfg = normalize_rope_scaling(cfg)
        model = AutoModelForCausalLM.from_pretrained(CFG.base_model, config=cfg, trust_remote_code=False, **load_kwargs)
    except Exception as e:
        log(f"Native load failed ({type(e).__name__}: {e}); retrying with remote code.")
        cfg = AutoConfig.from_pretrained(CFG.base_model, trust_remote_code=True)
        cfg = normalize_rope_scaling(cfg)
        model = AutoModelForCausalLM.from_pretrained(CFG.base_model, config=cfg, trust_remote_code=True, **load_kwargs)
    model.config.use_cache = False
    model.eval()
    return model


def attach_lora(model):
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=CFG.gradient_checkpointing)
    targets = discover_lora_targets(model)
    log(f"LoRA targets: {targets}")
    lcfg = LoraConfig(
        r=CFG.lora_r,
        lora_alpha=CFG.lora_alpha,
        lora_dropout=CFG.lora_dropout,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=targets,
    )
    model = get_peft_model(model, lcfg)
    if CFG.gradient_checkpointing and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    model.print_trainable_parameters()
    return model


def load_policy_model(tokenizer):
    device = POLICY_DEVICE
    base = load_base_model(device)
    if CFG.start_adapter and os.path.isdir(CFG.start_adapter) and os.path.exists(os.path.join(CFG.start_adapter, "adapter_config.json")):
        log(f"Loading starting adapter: {CFG.start_adapter}")
        try:
            # FIX: Adding is_trainable=True is vital for enabling backprop on the adapter.
            base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=CFG.gradient_checkpointing)
            base = PeftModel.from_pretrained(base, CFG.start_adapter, is_trainable=True)
            if CFG.gradient_checkpointing and hasattr(base, "gradient_checkpointing_enable"):
                base.gradient_checkpointing_enable()
            base.print_trainable_parameters()
        except Exception as e:
            log(f"Could not load start adapter ({type(e).__name__}: {e}); continuing from base.")
            base = attach_lora(base)
    else:
        base = attach_lora(base)
    base.train()
    return base


# ============================================================
# Tokenization and logprobs
# ============================================================

def tokenize_text(tokenizer, text: str, device: torch.device):
    out = tokenizer(text, return_tensors="pt", truncation=True, max_length=CFG.max_prompt_tokens)
    return {k: v.to(device) for k, v in out.items()}


def build_input_ids(tokenizer, prompt: str, completion: str, device: torch.device):
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CFG.max_prompt_tokens)["input_ids"][0]
    comp_ids = tokenizer(completion, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    input_ids = torch.cat([prompt_ids, comp_ids], dim=0).unsqueeze(0).to(device)
    labels = torch.full_like(input_ids, -100)
    labels[0, prompt_ids.shape[0]:] = comp_ids.to(device)
    attention_mask = torch.ones_like(input_ids)
    return input_ids, attention_mask, labels, prompt_ids.shape[0], comp_ids


def sequence_logprob(model, input_ids, attention_mask, labels):
    out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, output_hidden_states=False, return_dict=True)
    # HuggingFace returns mean loss over active tokens.
    # We can recover sum-logprob approx by multiplying with number of supervised tokens.
    supervised = int((labels != -100).sum().item())
    return -out.loss * supervised, out.loss, out.logits


def completion_token_logprobs(logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    # logits: [B, T, V], labels: [B, T]
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()
    log_probs = F.log_softmax(shift_logits, dim=-1)
    gather = torch.gather(log_probs, dim=-1, index=shift_labels.clamp(min=0).unsqueeze(-1)).squeeze(-1)
    mask = shift_labels.ne(-100).float()
    return (gather * mask).sum(dim=-1)


# ============================================================
# Layerwise routing loss (TLens-inspired)
# ============================================================

def final_norm_and_head(model):
    core = getattr(model, "model", model)
    norm = getattr(core, "norm", None)
    if norm is None:
        norm = getattr(core, "final_layer_norm", None)
    if norm is None:
        norm = getattr(model, "norm", None)
    head = getattr(model, "lm_head", None)
    if head is None:
        head = getattr(model, "embed_out", None)
    if head is None:
        raise RuntimeError("Could not locate final norm / lm_head.")
    return norm, head


def layerwise_answer_routing_loss(model, hidden_states, answer_pos: int, target_ids: List[int]) -> torch.Tensor:
    """Encourage the gold answer token to become salient only in late layers.

    This is the TLens-inspired part: the gold answer token should stay weak early,
    then rise sharply in the later residual stream.
    """
    if hidden_states is None or len(hidden_states) < 3:
        return torch.tensor(0.0, device=next(model.parameters()).device)
    norm, head = final_norm_and_head(model)
    device = hidden_states[0].device
    num_layers = len(hidden_states)
    layers = torch.arange(num_layers, device=device).float()
    pivot = 0.72 * (num_layers - 1)
    tau = max(1.5, 0.12 * num_layers)
    target_curve = torch.sigmoid((layers - pivot) / tau)  # late rise

    probs = []
    for h in hidden_states:
        v = h[:, answer_pos, :]
        if norm is not None:
            v = norm(v)
        lg = head(v)
        p = torch.softmax(lg.float(), dim=-1)
        if len(target_ids) > 0:
            probs.append(max(p[0, i].item() if i < p.shape[-1] else 0.0 for i in target_ids))
        else:
            probs.append(float(p.max().item()))
    probs = torch.tensor(probs, device=device)
    # Encourage monotonic late growth, but keep early layers low.
    mse = F.mse_loss(probs, target_curve)
    mono = torch.relu(probs[:-1] - probs[1:]).mean() if len(probs) > 1 else torch.tensor(0.0, device=device)
    return mse + 0.2 * mono


# ============================================================
# Completion generation / scoring
# ============================================================

@torch.no_grad()
def generate_one(model, tokenizer, prompt: str, max_new_tokens: int, temperature: float, top_p: float, do_sample: bool = True):
    inputs = tokenize_text(tokenizer, prompt, next(model.parameters()).device)
    
    # Clean up gen_kwargs to remove warning when do_sample=False
    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "do_sample": do_sample,
        "repetition_penalty": CFG.repetition_penalty,
    }
    
    if do_sample:
        gen_kwargs["temperature"] = max(0.1, temperature)
        gen_kwargs["top_p"] = top_p
        
    out = model.generate(**inputs, **gen_kwargs)
    
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return full, completion, out[0], inputs["input_ids"].shape[1]


def task_max_new_tokens(task: str) -> int:
    return CFG.max_new_gsm8k if task == "gsm8k" else CFG.max_new_qa if task == "strategyqa" else CFG.max_new_mmlu


def task_self_consistency(task: str) -> int:
    return CFG.self_consistency_gsm8k if task == "gsm8k" else CFG.self_consistency_qa if task == "strategyqa" else CFG.self_consistency_mmlu


def task_eval_max_new(task: str) -> int:
    return task_max_new_tokens(task)


def difficulty_weight(sample: Dict[str, Any], hard_df: pd.DataFrame) -> float:
    if len(hard_df) == 0:
        return 1.0
    sid = sample.get("sample_id")
    rows = hard_df[hard_df["sample_id"] == sid]
    if len(rows) > 0:
        return 1.0 + float(rows.iloc[0]["hard_score"])
    if sample["task"] == "strategyqa":
        return 1.25
    if sample["task"] == "mmlu":
        return 1.20
    return 1.0


def score_completion(sample: Dict[str, Any], completion: str, pred: Optional[str]) -> Dict[str, float]:
    task = sample["task"]
    format_ok = answer_present(completion)
    hard_bonus = 0.0
    if sample.get("hard_score", 0.0) > 0:
        hard_bonus = min(0.3, 0.15 + 0.15 * float(sample.get("hard_score", 0.0)))

    if task == "gsm8k":
        gold = canonical_number(sample["gold"])
        p = canonical_number(pred)
        exact = 1.0 if (p is not None and gold is not None and is_number_correct(p, gold)) else 0.0
        numeric_reward = 1.0 if exact else 0.0
        format_reward = 0.15 if format_ok else -0.1
        length_penalty = -0.001 * max(0, len(completion.split()) - 80)
        return {"reward": numeric_reward + format_reward + hard_bonus + length_penalty, "exact": exact, "format": float(format_ok)}

    if task == "strategyqa":
        exact = 1.0 if (pred == sample["gold"]) else 0.0
        yes_bias_fix = 0.06 if (sample["gold"] == "yes" and pred == "yes") else 0.0
        format_reward = 0.10 if format_ok else -0.10
        return {"reward": exact + yes_bias_fix + format_reward + hard_bonus, "exact": exact, "format": float(format_ok)}

    if task == "mmlu":
        exact = 1.0 if (pred == sample["gold"]) else 0.0
        anchor_bonus = 0.10 if exact else -0.05 if pred == "A" else 0.0
        format_reward = 0.10 if format_ok else -0.10
        return {"reward": exact + anchor_bonus + format_reward + hard_bonus, "exact": exact, "format": float(format_ok)}

    return {"reward": 0.0, "exact": 0.0, "format": 0.0}


# ============================================================
# Data item builders
# ============================================================

def make_example_row(sample: Dict[str, Any], prompt: str, completion: str, target: str, fewshots: List[Dict[str, Any]], hard_weight: float = 1.0) -> Dict[str, Any]:
    return {
        "task": sample["task"],
        "subject": sample.get("subject", sample["task"]),
        "sample_id": sample["sample_id"],
        "question": sample["question"],
        "gold": sample["gold"],
        "prompt": prompt,
        "completion": completion,
        "target": target,
        "fewshot_count": len(fewshots),
        "hard_weight": hard_weight,
    }


# ============================================================
# Collation
# ============================================================

def pad_tensor_list(tensors: List[torch.Tensor], pad_value: int) -> torch.Tensor:
    max_len = max(t.shape[0] for t in tensors)
    out = []
    for t in tensors:
        if t.shape[0] < max_len:
            pad = torch.full((max_len - t.shape[0],), pad_value, dtype=t.dtype, device=t.device)
            t = torch.cat([t, pad], dim=0)
        out.append(t)
    return torch.stack(out, dim=0)


def collate_supervised(batch: List[Dict[str, Any]], tokenizer, device: torch.device):
    input_ids, attn, labels, answer_pos, target_ids = [], [], [], [], []
    meta = []
    for ex in batch:
        ids = tokenizer(ex["prompt"], return_tensors="pt", truncation=True, max_length=CFG.max_prompt_tokens)["input_ids"][0]
        comp = tokenizer(ex["completion"], add_special_tokens=False, return_tensors="pt")["input_ids"][0]
        seq = torch.cat([ids, comp], dim=0)
        lab = torch.full_like(seq, -100)
        lab[ids.shape[0]:] = comp
        input_ids.append(seq.to(device))
        attn.append(torch.ones_like(seq, device=device))
        labels.append(lab.to(device))
        answer_pos.append(ids.shape[0] + max(0, comp.shape[0] - 1))
        target_ids.append(comp.tolist())
        meta.append(ex)
    return {
        "input_ids": pad_tensor_list(input_ids, tokenizer.pad_token_id),
        "attention_mask": pad_tensor_list(attn, 0),
        "labels": pad_tensor_list(labels, -100),
        "answer_pos": answer_pos,
        "target_ids": target_ids,
        "meta": meta,
    }


# ============================================================
# Training losses
# ============================================================

def supervised_loss(model, batch, tokenizer):
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
        output_hidden_states=True,
        return_dict=True,
    )
    ce = out.loss
    route_losses = []
    margin_losses = []
    for i, meta in enumerate(batch["meta"]):
        ans_pos = min(batch["answer_pos"][i], out.hidden_states[0].shape[1] - 1)
        target_txt = meta["gold"] if meta["task"] != "gsm8k" else meta["gold"]
        gold_ids = choice_ids(tokenizer, str(target_txt))
        route_losses.append(layerwise_answer_routing_loss(model, [h[i:i+1] for h in out.hidden_states], ans_pos, gold_ids))
        # final margin loss on answer position
        norm, head = final_norm_and_head(model)
        v = out.hidden_states[-1][i:i+1, ans_pos:ans_pos+1, :]
        if norm is not None:
            v = norm(v)
        logits = head(v)[0, 0].float()
        if gold_ids:
            gold_log = torch.max(logits[gold_ids])
            top_non = torch.topk(logits, k=min(8, logits.shape[-1]))
            top_non = top_non.values[0] if len(top_non.values) else logits.max()
            margin_losses.append(F.softplus(-(gold_log - top_non)))
    route = torch.stack(route_losses).mean() if route_losses else torch.tensor(0.0, device=ce.device)
    margin = torch.stack(margin_losses).mean() if margin_losses else torch.tensor(0.0, device=ce.device)
    total = CFG.w_supervised * ce + CFG.w_routing * route + CFG.w_margin * margin
    return total, {"ce": float(ce.item()), "route": float(route.item()), "margin": float(margin.item())}


def pairwise_rank_loss(model, tokenizer, chosen_batch, rejected_batch):
    # chosen/rejected are already prompt+completion concatenations.
    ch = model(input_ids=chosen_batch["input_ids"], attention_mask=chosen_batch["attention_mask"], labels=chosen_batch["labels"], return_dict=True)
    rj = model(input_ids=rejected_batch["input_ids"], attention_mask=rejected_batch["attention_mask"], labels=rejected_batch["labels"], return_dict=True)
    # Approx sequence logprob under token-level CE.
    ch_sup = (chosen_batch["labels"] != -100).sum(dim=1).float().clamp(min=1)
    rj_sup = (rejected_batch["labels"] != -100).sum(dim=1).float().clamp(min=1)
    ch_lp = -ch.loss * ch_sup.mean()
    rj_lp = -rj.loss * rj_sup.mean()
    diff = ch_lp - rj_lp
    loss = -F.logsigmoid(diff * 0.5)
    return loss, {"chosen_lp": float(ch_lp.item()), "rejected_lp": float(rj_lp.item()), "pair_diff": float(diff.item())}


# ============================================================
# Dataset builders for training phases
# ============================================================

def attach_prompt_and_target(df: pd.DataFrame, tokenizer, fewshot_bank: Dict[str, Any], hard_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in df.iterrows():
        sample = r.to_dict()
        if sample["task"] == "mmlu" and not isinstance(sample.get("choices"), list):
            # If hard-csv sample has no choices, keep a generic template only for pairwise prompts.
            sample["choices"] = ["A", "B", "C", "D"]
        fs = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if sample["task"] == "gsm8k" else CFG.fewshot_qa if sample["task"] == "strategyqa" else CFG.fewshot_mmlu)
        prompt = render_prompt(tokenizer, sample, fs)
        completion = render_target_completion(sample)
        
        sample["prompt"] = prompt
        sample["completion"] = completion
        sample["target"] = completion
        sample["fewshot_count"] = len(fs)
        sample["hard_weight"] = difficulty_weight(sample, hard_df)
        rows.append(sample)
    return pd.DataFrame(rows)


# ============================================================
# Evaluation / generation
# ============================================================

def eval_task_pool(model, tokenizer, df: pd.DataFrame, task: str, self_consistency: int, max_new_tokens: int, fewshot_bank: Dict[str, Any]):
    if len(df) == 0:
        return pd.DataFrame(), 0.0
    rows = []
    
    with evaluate_mode(model):
        for idx, r in df.iterrows():
            if idx % 10 == 0 or idx == len(df) - 1:
                log(f"Evaluating {task}: {idx+1}/{len(df)}")
                
            sample = r.to_dict()
            fs = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if task == "gsm8k" else CFG.fewshot_qa if task == "strategyqa" else CFG.fewshot_mmlu)
            prompt = render_prompt(tokenizer, sample, fs)
            completions = []
            preds = []
            scores = []
            temp = 0.7 if task != "gsm8k" else 0.8
            for _ in range(self_consistency):
                _, comp, _, _ = generate_one(model, tokenizer, prompt, max_new_tokens=max_new_tokens, temperature=temp, top_p=0.92, do_sample=True)
                pred, correct = parse_prediction(sample, comp)
                sc = score_completion(sample, comp, pred)
                completions.append(comp)
                preds.append(pred)
                scores.append(sc)
            # Majority vote / best-score fallback.
            valid_preds = [p for p in preds if p is not None]
            if task == "gsm8k":
                # numeric vote
                cnt = {}
                for p in valid_preds:
                    cnt[p] = cnt.get(p, 0) + 1
                if cnt:
                    pred = sorted(cnt.items(), key=lambda x: (x[1], x[0]), reverse=True)[0][0]
                else:
                    pred = preds[0]
            else:
                cnt = {}
                for p in valid_preds:
                    cnt[p] = cnt.get(p, 0) + 1
                pred = sorted(cnt.items(), key=lambda x: (x[1], str(x[0])), reverse=True)[0][0] if cnt else preds[0]
            correct = False
            if task == "gsm8k" and pred is not None and sample["gold"] is not None:
                correct = is_number_correct(pred, sample["gold"])
            elif task == "strategyqa":
                correct = pred == sample["gold"]
            elif task == "mmlu":
                correct = pred == sample["gold"]
            rows.append({
                **sample,
                "prediction": pred,
                "correct": bool(correct),
                "completion": completions[0],
                "self_consistency": self_consistency,
                "votes": json.dumps(cnt if task != "gsm8k" else cnt),
                "avg_reward": float(np.mean([x["reward"] for x in scores])) if scores else 0.0,
            })
    out = pd.DataFrame(rows)
    return out, float(out["correct"].mean())


# ============================================================
# Checkpoint / plots
# ============================================================

def save_adapter(model, tokenizer, stage_name: str):
    path = os.path.join(CKPT_DIR, stage_name)
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    log(f"Saved adapter checkpoint -> {path}")
    return path


# ============================================================
# Training: SFT warmup
# ============================================================

def train_sft_stage(model, tokenizer, train_df: pd.DataFrame, eval_sets: Dict[str, pd.DataFrame], hard_df: pd.DataFrame, fewshot_bank: Dict[str, Any]):
    log("===== SFT WARMUP STAGE =====")
    model.train()
    optim = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.wd)
    hist = []
    task_probs = task_sampling_probs(train_df, hard_df)
    
    # We must explicitly attach rendered prompts/targets to the SFT training rows.
    raw_order = sample_training_rows(train_df, task_probs, CFG.sft_steps * CFG.micro_batch_size)
    order_df = pd.DataFrame(raw_order)
    order_df = attach_prompt_and_target(order_df, tokenizer, fewshot_bank, hard_df)
    order = order_df.to_dict("records")

    for step in range(CFG.sft_steps):
        batch_rows = [order[(step * CFG.micro_batch_size + i) % len(order)] for i in range(CFG.micro_batch_size)]
        batch = collate_supervised(batch_rows, tokenizer, POLICY_DEVICE)
        loss, metrics = supervised_loss(model, batch, tokenizer)
        loss = loss / CFG.grad_accum
        loss.backward()
        if (step + 1) % CFG.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optim.step(); optim.zero_grad(set_to_none=True)

        hist.append({"step": step, "loss": float(loss.item()), **metrics})
        if (step + 1) % CFG.eval_every == 0:
            eval_acc = evaluate_all(model, tokenizer, eval_sets, fewshot_bank, stage=f"sft_{step+1}")
            hist[-1]["eval_acc"] = eval_acc["macro"]
            hist[-1]["reward"] = eval_acc["macro"]
            log(f"[SFT] step {step+1} | loss={metrics['ce']:.4f} | eval={eval_acc['macro']:.3f}")
        if (step + 1) % CFG.save_every == 0:
            save_adapter(model, tokenizer, f"sft_step_{step+1}")

    hist_df = pd.DataFrame(hist)
    save_df(hist_df, os.path.join(CSV_DIR, "sft_history.csv"))
    plot_training_curves(hist_df, os.path.join(PLOT_DIR, "sft_curves.png"))
    save_adapter(model, tokenizer, "sft_final")
    return model, hist_df


# ============================================================
# Training: Pairwise preference optimization (DPO-like, ref-free)
# ============================================================

def sample_training_rows(train_df: pd.DataFrame, task_probs: Dict[str, float], n: int) -> List[Dict[str, Any]]:
    if len(train_df) == 0:
        return []
    task_groups = {t: train_df[train_df.task == t].sample(min(len(train_df[train_df.task == t]), max(1, int(n * task_probs.get(t, 1/3)))), replace=len(train_df[train_df.task == t]) < 1, random_state=CFG.seed).to_dict("records") for t in train_df.task.unique()}
    rows = []
    while len(rows) < n:
        for t, items in task_groups.items():
            if items:
                rows.append(random.choice(items))
                if len(rows) >= n:
                    break
    random.shuffle(rows)
    return rows


def task_sampling_probs(train_df: pd.DataFrame, hard_df: pd.DataFrame) -> Dict[str, float]:
    if len(train_df) == 0:
        return {"gsm8k": 1/3, "strategyqa": 1/3, "mmlu": 1/3}
    base = {t: 1.0 - float(train_df[train_df.task == t]["gold"].isna().mean()) for t in train_df.task.unique()}
    # Weight harder tasks more aggressively.
    if len(train_df[train_df.task == "gsm8k"]):
        base["gsm8k"] = 1.0
    if len(train_df[train_df.task == "strategyqa"]):
        base["strategyqa"] = 1.5
    if len(train_df[train_df.task == "mmlu"]):
        base["mmlu"] = 1.35
    s = sum(base.values())
    return {k: v / s for k, v in base.items()}


def build_preference_pairs(model, tokenizer, train_df: pd.DataFrame, hard_df: pd.DataFrame, fewshot_bank: Dict[str, Any], n_prompts: int = 120):
    log("Building preference pairs from current model rollouts ...")
    prompt_pool = sample_training_rows(train_df, task_sampling_probs(train_df, hard_df), n_prompts)
    pairs = []
    
    with evaluate_mode(model):
        for i, sample in enumerate(prompt_pool):
            fs = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if sample["task"] == "gsm8k" else CFG.fewshot_qa if sample["task"] == "strategyqa" else CFG.fewshot_mmlu)
            prompt = render_prompt(tokenizer, sample, fs)
            cand_rows = []
            temp = CFG.temperature_start
            for c in range(CFG.num_candidates):
                full, comp, toks, p_len = generate_one(
                    model,
                    tokenizer,
                    prompt,
                    task_max_new_tokens(sample["task"]),
                    temperature=max(0.35, temp - 0.08 * c),
                    top_p=CFG.top_p,
                    do_sample=True,
                )
                pred, _ = parse_prediction(sample, comp)
                sc = score_completion(sample, comp, pred)
                cand_rows.append({
                    "prompt": prompt,
                    "completion": comp,
                    "prediction": pred,
                    "reward": sc["reward"],
                    "exact": sc["exact"],
                    "format": sc["format"],
                    "sample_id": sample["sample_id"],
                    "task": sample["task"],
                    "gold": sample["gold"],
                    "question": sample["question"],
                })
            cand_rows = sorted(cand_rows, key=lambda x: (x["reward"], x["exact"], x["format"]), reverse=True)
            if len(cand_rows) >= 2 and cand_rows[0]["reward"] - cand_rows[-1]["reward"] > 0.15:
                chosen = cand_rows[0]
                rejected = cand_rows[-1]
                pairs.append({
                    "prompt": chosen["prompt"],
                    "chosen": chosen["completion"],
                    "rejected": rejected["completion"],
                    "task": chosen["task"],
                    "sample_id": chosen["sample_id"],
                    "chosen_reward": chosen["reward"],
                    "rejected_reward": rejected["reward"],
                    "gold": chosen["gold"],
                    "question": chosen["question"],
                })
                
    pref_df = pd.DataFrame(pairs)
    if len(pref_df) == 0:
        log("No preference pairs mined; using a small fallback from supervised targets.")
        for _, sample in pd.concat([train_df.head(20)]).iterrows():
            fs = choose_fewshots(sample.to_dict(), fewshot_bank, CFG.fewshot_gsm8k if sample.task == "gsm8k" else CFG.fewshot_qa if sample.task == "strategyqa" else CFG.fewshot_mmlu)
            prompt = render_prompt(tokenizer, sample.to_dict(), fs)
            chosen = render_target_completion(sample.to_dict())
            rejected = "I am not sure." if sample.task != "mmlu" else "<answer>A</answer>"
            pref_df = pd.concat([pref_df, pd.DataFrame([{
                "prompt": prompt,
                "chosen": chosen,
                "rejected": rejected,
                "task": sample.task,
                "sample_id": sample.sample_id,
                "chosen_reward": 1.0,
                "rejected_reward": 0.0,
                "gold": sample.gold,
                "question": sample.question,
            }])], ignore_index=True)
    save_df(pref_df, os.path.join(CSV_DIR, "preference_pairs.csv"))
    return pref_df


def train_pairwise_stage(model, tokenizer, pref_df: pd.DataFrame, fewshot_bank: Dict[str, Any], eval_sets: Dict[str, pd.DataFrame]):
    log("===== PAIRWISE PREFERENCE STAGE =====")
    model.train()
    optim = torch.optim.AdamW(model.parameters(), lr=CFG.lr * 0.6, weight_decay=CFG.wd)
    hist = []
    rows = pref_df.sample(frac=1.0, random_state=CFG.seed).to_dict("records")
    if len(rows) == 0:
        return model, pd.DataFrame()
    for step in range(CFG.pairwise_steps):
        r = rows[step % len(rows)]
        sample = {"task": r["task"], "sample_id": r["sample_id"], "question": r["question"], "gold": r["gold"], "choices": ["A", "B", "C", "D"]}
        fs = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if sample["task"] == "gsm8k" else CFG.fewshot_qa if sample["task"] == "strategyqa" else CFG.fewshot_mmlu)
        prompt = r["prompt"]
        ch_ids, ch_mask, ch_labels, _, _ = build_input_ids(tokenizer, prompt, r["chosen"], POLICY_DEVICE)
        rj_ids, rj_mask, rj_labels, _, _ = build_input_ids(tokenizer, prompt, r["rejected"], POLICY_DEVICE)
        loss, m = pairwise_rank_loss(model, tokenizer, {"input_ids": ch_ids, "attention_mask": ch_mask, "labels": ch_labels}, {"input_ids": rj_ids, "attention_mask": rj_mask, "labels": rj_labels})
        loss.backward()
        if (step + 1) % CFG.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optim.step(); optim.zero_grad(set_to_none=True)
        hist.append({"step": step, "loss": float(loss.item()), **m})
        if (step + 1) % CFG.eval_every == 0:
            eval_acc = evaluate_all(model, tokenizer, eval_sets, fewshot_bank, stage=f"pair_{step+1}")
            hist[-1]["eval_acc"] = eval_acc["macro"]
            hist[-1]["reward"] = eval_acc["macro"]
            log(f"[PAIR] step {step+1} | loss={loss.item():.4f} | eval={eval_acc['macro']:.3f}")
        if (step + 1) % CFG.save_every == 0:
            save_adapter(model, tokenizer, f"pair_step_{step+1}")
    hist_df = pd.DataFrame(hist)
    save_df(hist_df, os.path.join(CSV_DIR, "pairwise_history.csv"))
    plot_training_curves(hist_df, os.path.join(PLOT_DIR, "pairwise_curves.png"))
    save_adapter(model, tokenizer, "pairwise_final")
    return model, hist_df


# ============================================================
# Online RRPO / GRPO-like stage (ref-free)
# ============================================================

def answer_token_target(sample: Dict[str, Any], tokenizer):
    if sample["task"] == "gsm8k":
        return choice_ids(tokenizer, str(sample["gold"]))
    if sample["task"] == "strategyqa":
        return choice_ids(tokenizer, sample["gold"])
    if sample["task"] == "mmlu":
        return choice_ids(tokenizer, sample["gold"])
    return []


def score_rollout(sample: Dict[str, Any], completion: str, pred: Optional[str], hard_df: pd.DataFrame) -> Dict[str, float]:
    base = score_completion(sample, completion, pred)
    # TLens-informed reward shaping: favor a clean answer tag and strong final answer routing.
    format_bonus = 0.0
    if answer_present(completion):
        format_bonus = 0.08
    hard = difficulty_weight(sample, hard_df)
    reward = base["reward"] + format_bonus + 0.05 * hard
    return {**base, "reward": reward}


def online_rrpo_stage(model, tokenizer, train_df: pd.DataFrame, hard_df: pd.DataFrame, fewshot_bank: Dict[str, Any], eval_sets: Dict[str, pd.DataFrame]):
    log("===== ONLINE RRPO STAGE =====")
    model.train()
    optim = torch.optim.AdamW(model.parameters(), lr=CFG.lr * 0.5, weight_decay=CFG.wd)
    hist = []
    task_probs = task_sampling_probs(train_df, hard_df)
    prompt_pool = sample_training_rows(train_df, task_probs, CFG.rrpo_steps)
    if len(prompt_pool) == 0:
        return model, pd.DataFrame()

    for step in range(CFG.rrpo_steps):
        sample = prompt_pool[step % len(prompt_pool)]
        fs = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if sample["task"] == "gsm8k" else CFG.fewshot_qa if sample["task"] == "strategyqa" else CFG.fewshot_mmlu)
        prompt = render_prompt(tokenizer, sample, fs)
        # Generate group.
        rollouts = []
        temp = max(CFG.temperature_end, CFG.temperature_start - (CFG.temperature_start - CFG.temperature_end) * (step / max(1, CFG.rrpo_steps - 1)))
        
        with evaluate_mode(model):
            for c in range(CFG.num_candidates):
                full, comp, gen_ids, prompt_len = generate_one(
                    model, tokenizer, prompt, task_max_new_tokens(sample["task"]),
                    temperature=max(0.35, temp - 0.05 * c), top_p=CFG.top_p, do_sample=True
                )
                pred, _ = parse_prediction(sample, comp)
                sc = score_rollout(sample, comp, pred, hard_df)
                # Compute old logprob of the rollout under the current policy.
                # Build exact sequence and store sequence-level logprob for PPO-style ratio.
                input_ids, attn, labels, _, _ = build_input_ids(tokenizer, prompt, comp, POLICY_DEVICE)
                with torch.no_grad():
                    lp, ce, logits = sequence_logprob(model, input_ids, attn, labels)
                    entropy = float(-(F.softmax(logits[0, -1].float(), dim=-1) * F.log_softmax(logits[0, -1].float(), dim=-1)).sum().item())
                rollouts.append({
                    "prompt": prompt,
                    "completion": comp,
                    "prediction": pred,
                    "reward": sc["reward"],
                    "exact": sc["exact"],
                    "format": sc["format"],
                    "seq_logp": float(lp.item()),
                    "entropy": entropy,
                    "input_ids": input_ids,
                    "attention_mask": attn,
                    "labels": labels,
                    "sample": sample,
                    "hard_weight": difficulty_weight(sample, hard_df),
                    "answer_pos": input_ids.shape[1] - 1,
                })

        rewards = torch.tensor([r["reward"] for r in rollouts], dtype=torch.float32, device=POLICY_DEVICE)
        adv = (rewards - rewards.mean())
        if rewards.std() > 1e-6:
            adv = adv / (rewards.std() + 1e-6)

        # Policy gradient update.
        pg_losses = []
        entropies = []
        route_losses = []
        margin_losses = []
        for i, r in enumerate(rollouts):
            out = model(input_ids=r["input_ids"], attention_mask=r["attention_mask"], labels=r["labels"], output_hidden_states=True, return_dict=True)
            curr_seq_lp = -out.loss * max(1, int((r["labels"] != -100).sum().item()))
            ratio = torch.exp(curr_seq_lp - torch.tensor(r["seq_logp"], device=POLICY_DEVICE))
            clipped = torch.clamp(ratio, 1.0 - CFG.clip_eps, 1.0 + CFG.clip_eps)
            pg = -torch.min(ratio * adv[i], clipped * adv[i])
            pg_losses.append(pg)
            entropies.append(torch.tensor(r["entropy"], device=POLICY_DEVICE))

            # Routing regularizer on the sampled completion.
            tgt_ids = answer_token_target(r["sample"], tokenizer)
            route_losses.append(layerwise_answer_routing_loss(model, list(out.hidden_states), r["answer_pos"], tgt_ids))

            # Final answer margin on the sampled completion.
            norm, head = final_norm_and_head(model)
            v = out.hidden_states[-1][:, r["answer_pos"], :]
            if norm is not None:
                v = norm(v)
            lg = head(v)[0].float()
            if tgt_ids:
                gold_log = torch.max(lg[tgt_ids])
                top_non = torch.topk(lg, k=min(8, lg.shape[-1])).values[0]
                margin_losses.append(F.softplus(-(gold_log - top_non)))
            else:
                margin_losses.append(torch.tensor(0.0, device=POLICY_DEVICE))

        loss = torch.stack(pg_losses).mean() + CFG.w_rl_entropy * (-torch.stack(entropies).mean()) + CFG.w_routing * torch.stack(route_losses).mean() + CFG.w_reward_margin * torch.stack(margin_losses).mean()
        loss.backward()
        if (step + 1) % CFG.grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            optim.step(); optim.zero_grad(set_to_none=True)

        hist.append({
            "step": step,
            "loss": float(loss.item()),
            "reward": float(rewards.mean().item()),
            "adv": float(adv.mean().item()),
            "entropy": float(np.mean([r["entropy"] for r in rollouts])),
            "route": float(torch.stack(route_losses).mean().item()),
            "margin": float(torch.stack(margin_losses).mean().item()),
            "exact_rate": float(np.mean([r["exact"] for r in rollouts])),
            "format_rate": float(np.mean([r["format"] for r in rollouts])),
        })

        if (step + 1) % CFG.eval_every == 0:
            eval_acc = evaluate_all(model, tokenizer, eval_sets, fewshot_bank, stage=f"rrpo_{step+1}")
            hist[-1]["eval_acc"] = eval_acc["macro"]
            log(f"[RRPO] step {step+1} | loss={loss.item():.4f} | reward={hist[-1]['reward']:.3f} | eval={eval_acc['macro']:.3f}")
        if (step + 1) % CFG.save_every == 0:
            save_adapter(model, tokenizer, f"rrpo_step_{step+1}")

    hist_df = pd.DataFrame(hist)
    save_df(hist_df, os.path.join(CSV_DIR, "rrpo_history.csv"))
    plot_training_curves(hist_df, os.path.join(PLOT_DIR, "rrpo_curves.png"))
    save_adapter(model, tokenizer, "rrpo_final")
    return model, hist_df


# ============================================================
# Evaluation wrappers
# ============================================================

def evaluate_all(model, tokenizer, eval_sets: Dict[str, pd.DataFrame], fewshot_bank: Dict[str, Any], stage: str = "eval"):
    out_rows = []
    metrics = {}
    with evaluate_mode(model):
        for task, df in [("gsm8k", eval_sets.get("gsm_eval", pd.DataFrame())), ("strategyqa", eval_sets.get("qa_eval", pd.DataFrame())), ("mmlu", eval_sets.get("mmlu_eval", pd.DataFrame()))]:
            if len(df) == 0:
                metrics[task] = 0.0
                continue
            self_c = task_self_consistency(task)
            max_new = task_eval_max_new(task)
            rows = []
            for i, (_, r) in enumerate(df.iterrows()):
                if i % 10 == 0 or i == len(df) - 1:
                    log(f"[{stage}] Evaluating {task}: {i+1}/{len(df)}")
                sample = r.to_dict()
                fs = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if task == "gsm8k" else CFG.fewshot_qa if task == "strategyqa" else CFG.fewshot_mmlu)
                prompt = render_prompt(tokenizer, sample, fs)
                preds = []
                comps = []
                rewards = []
                for _ in range(self_c):
                    full, comp, _, _ = generate_one(model, tokenizer, prompt, max_new, temperature=0.72 if task != "gsm8k" else 0.82, top_p=0.92, do_sample=True)
                    pred, _ = parse_prediction(sample, comp)
                    sc = score_completion(sample, comp, pred)
                    preds.append(pred)
                    comps.append(comp)
                    rewards.append(sc["reward"])
                # vote
                cnt = {}
                for p in preds:
                    if p is None:
                        continue
                    cnt[p] = cnt.get(p, 0) + 1
                if len(cnt) > 0:
                    pred = sorted(cnt.items(), key=lambda x: (x[1], str(x[0])), reverse=True)[0][0]
                else:
                    pred = preds[0]
                correct = False
                if task == "gsm8k":
                    correct = is_number_correct(pred, sample["gold"])
                else:
                    correct = pred == sample["gold"]
                rows.append({
                    **sample,
                    "stage": stage,
                    "prediction": pred,
                    "correct": bool(correct),
                    "completion": comps[0],
                    "votes": json.dumps(cnt),
                    "avg_reward": float(np.mean(rewards)),
                    "self_consistency": self_c,
                    "model": stage,
                })
            out_df = pd.DataFrame(rows)
            save_df(out_df, os.path.join(CSV_DIR, f"{stage}_{task}_predictions.csv"))
            metrics[task] = float(out_df["correct"].mean())
            out_rows.append(out_df)
    metrics["macro"] = float(np.mean([metrics.get("gsm8k", 0.0), metrics.get("strategyqa", 0.0), metrics.get("mmlu", 0.0)]))
    return metrics


def evaluate_model_detailed(model, tokenizer, eval_sets: Dict[str, pd.DataFrame], fewshot_bank: Dict[str, Any], stage: str):
    rows = []
    task_metrics = {}
    with evaluate_mode(model):
        for task, df in [("gsm8k", eval_sets.get("gsm_eval", pd.DataFrame())), ("strategyqa", eval_sets.get("qa_eval", pd.DataFrame())), ("mmlu", eval_sets.get("mmlu_eval", pd.DataFrame()))]:
            if len(df) == 0:
                continue
            task_rows = []
            for i, (_, r) in enumerate(df.iterrows()):
                if i % 10 == 0 or i == len(df) - 1:
                    log(f"[{stage}] Detailed Eval {task}: {i+1}/{len(df)}")
                sample = r.to_dict()
                fs = choose_fewshots(sample, fewshot_bank, CFG.fewshot_gsm8k if task == "gsm8k" else CFG.fewshot_qa if task == "strategyqa" else CFG.fewshot_mmlu)
                prompt = render_prompt(tokenizer, sample, fs)
                full, comp, _, _ = generate_one(model, tokenizer, prompt, task_eval_max_new(task), temperature=0.0, top_p=1.0, do_sample=False)
                pred, correct = parse_prediction(sample, comp)
                sc = score_completion(sample, comp, pred)
                route = extract_route(comp)
                
                task_rows.append({
                    **sample,
                    "stage": stage,
                    "prediction": pred,
                    "correct": bool(correct),
                    "completion": comp,
                    "full_output": full,
                    "reward": sc["reward"],
                    "model": stage,
                    "route_pred": route,
                    "completion_length": len(comp)
                })
            task_df = pd.DataFrame(task_rows)
            rows.append(task_df)
            task_metrics[task] = float(task_df["correct"].mean())
            save_df(task_df, os.path.join(CSV_DIR, f"{stage}_{task}_deterministic.csv"))
    task_metrics["macro"] = float(np.mean(list(task_metrics.values()))) if task_metrics else 0.0
    return task_metrics, (pd.concat(rows, ignore_index=True) if rows else pd.DataFrame())


# ============================================================
# Main orchestration
# ============================================================

global tokenizer

def main():
    global tokenizer
    log("=======================================================")
    log("PHI-3 HARD-MINING + ROUTING-AWARE RRPO TRAINING")
    log("=======================================================")
    gpu_report("Startup GPU")

    with timer("Tokenizer load"):
        tokenizer = load_tokenizer()

    with timer("Task pool build"):
        pools = build_task_pools()

    # Mine failures from your CSV.
    error_df = mine_hard_examples(load_error_csv(CFG.hard_csv))
    if len(error_df) > 0:
        save_df(error_df, os.path.join(CSV_DIR, "hard_mined_examples.csv"))
        log(f"Hard mining found {len(error_df)} persistent / high-error examples.")
    else:
        log("No hard-mining CSV available or it was malformed; continuing without it.")

    # Train/eval pools (augment with hard failures).
    train_df = pd.concat([pools["gsm_train"], pools["qa_train"], pools["mmlu_train"]], ignore_index=True)
    train_df = add_hard_examples(train_df, error_df)
    fewshot_bank = build_fewshot_bank(train_df)
    eval_sets = {"gsm_eval": pools["gsm_eval"], "qa_eval": pools["qa_eval"], "mmlu_eval": pools["mmlu_eval"]}

    # Save evaluation baselines from the starting checkpoint.
    log("Loading policy model ...")
    with timer("Policy model load"):
        policy = load_policy_model(tokenizer)
    policy.to(POLICY_DEVICE)
    policy.train()
    gpu_report("After policy load")

    # Baseline eval before any extra training.
    with timer("Baseline eval"):
        baseline_metrics, baseline_det = evaluate_model_detailed(policy, tokenizer, eval_sets, fewshot_bank, stage="baseline")
    save_json(baseline_metrics, os.path.join(REPORT_DIR, "baseline_metrics.json"))
    save_df(baseline_det, os.path.join(CSV_DIR, "baseline_predictions.csv"))
    log(f"Baseline macro accuracy: {baseline_metrics['macro']:.3f}")

    progress_rows = []
    for task, acc in baseline_metrics.items():
        if task == "macro":
            continue
        progress_rows.append({"stage_idx": 0, "stage_name": "baseline", "task": task, "accuracy": acc})

    # Stage 1: routing-aware SFT warmup.
    policy, sft_hist = train_sft_stage(policy, tokenizer, train_df, eval_sets, error_df, fewshot_bank)
    sft_metrics, sft_det = evaluate_model_detailed(policy, tokenizer, eval_sets, fewshot_bank, stage="sft_final")
    save_json(sft_metrics, os.path.join(REPORT_DIR, "sft_metrics.json"))
    progress_rows.extend([
        {"stage_idx": 1, "stage_name": "sft_final", "task": k, "accuracy": v}
        for k, v in sft_metrics.items() if k != "macro"
    ])

    # Stage 2: pairwise preference optimization mined from self-rollouts.
    pref_df = build_preference_pairs(policy, tokenizer, train_df, error_df, fewshot_bank, n_prompts=120)
    policy, pref_hist = train_pairwise_stage(policy, tokenizer, pref_df, fewshot_bank, eval_sets)
    pref_metrics, pref_det = evaluate_model_detailed(policy, tokenizer, eval_sets, fewshot_bank, stage="pairwise_final")
    save_json(pref_metrics, os.path.join(REPORT_DIR, "pairwise_metrics.json"))
    progress_rows.extend([
        {"stage_idx": 2, "stage_name": "pairwise_final", "task": k, "accuracy": v}
        for k, v in pref_metrics.items() if k != "macro"
    ])

    # Stage 3: online RRPO / GRPO-like refinement.
    policy, rrpo_hist = online_rrpo_stage(policy, tokenizer, train_df, error_df, fewshot_bank, eval_sets)
    rrpo_metrics, rrpo_det = evaluate_model_detailed(policy, tokenizer, eval_sets, fewshot_bank, stage="rrpo_final")
    save_json(rrpo_metrics, os.path.join(REPORT_DIR, "rrpo_metrics.json"))
    progress_rows.extend([
        {"stage_idx": 3, "stage_name": "rrpo_final", "task": k, "accuracy": v}
        for k, v in rrpo_metrics.items() if k != "macro"
    ])

    # Save training histories and plots.
    save_df(sft_hist, os.path.join(CSV_DIR, "sft_history.csv"))
    save_df(pref_hist, os.path.join(CSV_DIR, "pairwise_history.csv"))
    save_df(rrpo_hist, os.path.join(CSV_DIR, "rrpo_history.csv"))
    plot_training_curves(sft_hist, os.path.join(PLOT_DIR, "sft_curves.png"))
    plot_training_curves(pref_hist, os.path.join(PLOT_DIR, "pairwise_curves.png"))
    plot_training_curves(rrpo_hist, os.path.join(PLOT_DIR, "rrpo_curves.png"))

    progress_df = pd.DataFrame(progress_rows)
    save_df(progress_df, os.path.join(CSV_DIR, "accuracy_progress.csv"))
    plot_accuracy_progress(progress_df, os.path.join(PLOT_DIR, "accuracy_progress.png"))

    # Deterministic final evaluation.
    final_metrics, final_det = evaluate_model_detailed(policy, tokenizer, eval_sets, fewshot_bank, stage="final_model")
    save_json(final_metrics, os.path.join(REPORT_DIR, "final_metrics.json"))
    save_df(final_det, os.path.join(CSV_DIR, "final_predictions.csv"))
    log(f"Final macro accuracy: {final_metrics['macro']:.3f}")
    
    # Generate final evaluation plots
    log("Generating final evaluation plots...")
    summary_df_for_plot = pd.DataFrame([{"task": k, "accuracy": v} for k, v in final_metrics.items() if k != "macro"])
    bar_plot(
        summary_df_for_plot["task"].tolist(), 
        summary_df_for_plot["accuracy"].tolist(), 
        "Final Model Accuracy by Task", 
        os.path.join(PLOT_DIR, "final_accuracy_by_task.png"),
        color_map="Set2"
    )
    
    if len(final_det) > 0 and "mmlu" in final_det["task"].values:
        mmlu_df = final_det[final_det["task"] == "mmlu"]
        if "subject" in mmlu_df.columns:
            subject_summary = mmlu_df.groupby("subject", as_index=False).agg(
                n=("correct", "count"),
                accuracy=("correct", "mean"),
            )
            bar_plot(
                subject_summary["subject"].tolist(), 
                subject_summary["accuracy"].tolist(), 
                "MMLU Final Accuracy by Subject", 
                os.path.join(PLOT_DIR, "final_mmlu_accuracy_by_subject.png"),
                rotate=15,
                color_map="Paired"
            )
        
    plot_route_distribution(final_det, os.path.join(PLOT_DIR, "final_route_distribution.png"), "Final Model Route Distribution")
    plot_length_hist(final_det, os.path.join(PLOT_DIR, "final_length_hist.png"), "Final Model Completion Lengths")
    violin_plot_lengths(final_det, os.path.join(PLOT_DIR, "final_completion_lengths_violin.png"))

    # Comparison table.
    comp = pd.DataFrame([
        {"stage": "baseline", **baseline_metrics},
        {"stage": "sft_final", **sft_metrics},
        {"stage": "pairwise_final", **pref_metrics},
        {"stage": "rrpo_final", **rrpo_metrics},
        {"stage": "final_model", **final_metrics},
    ])
    save_df(comp, os.path.join(CSV_DIR, "comparison_table.csv"))

    # Report.
    report_lines = [
        "# Phi-3 hard-mining + routing-aware RRPO report\n",
        f"- Base model: `{CFG.base_model}`",
        f"- Start adapter: `{CFG.start_adapter}`",
        f"- Hard CSV: `{CFG.hard_csv}`",
        f"- Device count: {DEVICE_COUNT}",
        f"- LoRA rank: {CFG.lora_r}",
        "\n## Baseline metrics\n",
        pd.DataFrame([baseline_metrics]).to_markdown(index=False),
        "\n## Final metrics\n",
        pd.DataFrame([final_metrics]).to_markdown(index=False),
        "\n## Comparison\n",
        comp.to_markdown(index=False),
        "\n## Visualizations\n",
        "![Task Accuracy](../plots/final_accuracy_by_task.png)\n",
        "![MMLU Accuracy by Subject](../plots/final_mmlu_accuracy_by_subject.png)\n",
        "![Route Distribution](../plots/final_route_distribution.png)\n",
        "![Completion Lengths](../plots/final_completion_lengths_violin.png)\n",
        "\n## Notes\n",
        "- Training uses hard-example mining from the uploaded CSV, with persistent failures oversampled.",
        "- The SFT stage includes a layerwise answer-routing regularizer that pushes answer salience late in the residual stream.",
        "- The pairwise stage is DPO-like but ref-free, which keeps it stable on Kaggle and avoids reference-model memory blowups.",
        "- The RRPO stage uses group-relative advantages and clipped policy ratios, mirroring GRPO-style online learning without depending on a brittle trainer API.",
        "- MMLU choice permutation augmentation is built in to fight positional A-bias.",
        "- Evaluation uses random sampling and self-consistency instead of first-N subsets.",
    ]
    with open(os.path.join(REPORT_DIR, "report.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))
    save_json({
        "config": asdict(CFG),
        "baseline": baseline_metrics,
        "final": final_metrics,
        "comparison": comp.to_dict(orient="records"),
    }, os.path.join(REPORT_DIR, "summary.json"))

    # Save final adapter.
    final_dir = os.path.join(CKPT_DIR, "final_model")
    os.makedirs(final_dir, exist_ok=True)
    policy.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    log(f"Saved final checkpoint -> {final_dir}")

    print("\n===== BASELINE =====")
    print(pd.DataFrame([baseline_metrics]).to_string(index=False))
    print("\n===== FINAL =====")
    print(pd.DataFrame([final_metrics]).to_string(index=False))
    print("\n===== COMPARISON =====")
    print(comp.to_string(index=False))
    print(f"\nOutputs saved to: {OUT_DIR}")

if __name__ == "__main__":
    main()

Installing missing dependency: bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00


[21:14:29] =======================================================


[21:14:29] PHI-3 HARD-MINING + ROUTING-AWARE RRPO TRAINING


[21:14:29] =======================================================


...

[09:13:17] Saved plot -> /kaggle/working/phi3_rrpo_hardmine/plots/pairwise_curves.png


[09:13:18] Saved plot -> /kaggle/working/phi3_rrpo_hardmine/plots/rrpo_curves.png


[09:13:18] Saved CSV -> /kaggle/working/phi3_rrpo_hardmine/csv/accuracy_progress.csv


[09:13:18] Saved plot -> /kaggle/working/phi3_rrpo_hardmine/plots/accuracy_progress.png


[09:13:18] [final_model] Detailed Eval gsm8k: 1/50
